# 02: Model W, the measuring part, and the record reader

Stages `model_w`, `estimator`, `record_lora` (blocks 1–3 in `docs/02_MODEL_STRUCTURE.md`).

**Plain.**
1. **Model W.** Teach the video-language model the question format on ordinary videos by training a thin "correction" on top of it (LoRA). The big model itself never changes.
2. **The measuring part.** A small separate network reads the model's view of each frame and says, for each ball, whether it is visible and where it is on the floor. Together with the known motion of the camera, this gives each ball's **estimated track**.
3. **The record reader.** A second LoRA teaches the model to answer from the **last frame plus one sentence** written by a calculator, for example "the red ball at (−1.20, +3.00) …", instead of from all 8 frames.

**Technical.**
- W: LoRA r = 16 on q, k, v, o of all 36 decoder layers; 1000 steps on train3 (`w1k_s0.pt`).
- Measuring part: a 2.45M-parameter transformer over the cached 64 tokens; 3000 steps on train3 + train8 (`state_estimator.pt`); tracks for every split (`<split>_est_tracks.json`).
- Record LoRA: from W, 1000 steps on train3 + train8 with `--text_state --hist last1` (`r1_lsq_s1.pt`).

In [ ]:
import sys; sys.path.insert(0, '.')     # nb_helpers.py sits next to this notebook
import nb_helpers as H
H.setup(quick=False, work=None)            # quick=True: minutes instead of hours, meaningless numbers
                                           # work='...': another workspace folder (default <repo>/workspace)

In [ ]:
H.run('model_w')       # ~9 min

In [ ]:
H.run('estimator')     # ~11 min

In [ ]:
H.compare('estimator')

In [ ]:
H.run('record_lora')   # ~8 min